Fitters search parameter values by repeatedly evaluating a loss. The
`FittingAlgorithm` protocol separates that search from the model factory and
objective, and `FitResult` describes the returned record.

## The fitting interface

Declaration from [`jaxcmr.typing`](https://github.com/githubpsyche/jaxcmr/blob/main/jaxcmr/typing.py):

```python
class FittingAlgorithm(Protocol):
    def __init__(
        self,
        dataset: RecallDataset,
        features: Float[Array, "word_pool_items features_count"] | None,
        base_params: Mapping[str, Float_],
        model_factory_cls: type[MemorySearchModelFactory],
        loss_fn_cls: type[LossFn],
        hyperparams: dict[str, Any] | None = None,
    ): ...

    def fit(
        self, trial_mask: Bool[Array, " trials"], subject_id: int = -1
    ) -> FitResult: ...

    def fit_subjects(self, trial_mask: Bool[Array, " trials"]) -> FitResult: ...
```

Existing implementations include `ScipyDE` and `EvosaxDE` in
`jaxcmr.fitting`. Both take a model factory class and a loss class. These
arguments are the entry points for changing the modeled process or fitting
objective without rewriting the optimization procedure. A compatible
alternative fitter supplies the same constructor and fitting methods;
optimizer-specific controls remain in its `hyperparams` mapping.

| Configuration | Meaning |
|---|---|
| `model_factory_cls` | Builds the models used by the loss. Component choices belong to this configuration. |
| `loss_fn_cls` | Defines the objective minimized by the search. |
| `base_params` | Fixed model values and defaults overridden by free parameters. |
| `hyperparams['bounds']` | Names of free parameters with lower and upper bounds in model units. |
| Other `hyperparams` | Search controls such as generations, population size, stopping tolerances and seed. |

Bounds define the free parameter order passed through the loss adapter.
Supported parameter transformations can change the search coordinates;
losses and returned fit values still use physical model units.

## Pooling and participant fits

`fit(trial_mask)` estimates one parameter set for all selected trials.
Its default result label is `subject_id=-1`, the pooled-fit convention.
`subject_id` labels the result; it does not itself select that participant's
trials. `fit_subjects(trial_mask)` performs separate fits using the dataset's
participant identifiers and the supplied selection.

A mask selects observations. Pooling specifies which observations share
parameters. Neither choice is determined by the optimizer. Changing from
pooled to participant fits changes the statistical model and the available
information per parameter set.

## The returned record

Declaration from [`jaxcmr.typing`](https://github.com/githubpsyche/jaxcmr/blob/main/jaxcmr/typing.py):

```python
class FitResult(TypedDict):
    fixed: dict[str, float]
    free: dict[str, list[float]]
    fitness: list[float]
    nit: list[int]
    converged: list[bool]
    fits: dict[str, list[float]]
    hyperparameters: dict[str, Any]
    fit_time: float
    diagnostics: NotRequired[dict[str, Any]]
```

Entries at the same index in `fitness`, `nit`, `converged` and every `fits`
array refer to the same fit row. `fits['subject']` identifies that row;
identifiers are labels, not positional indices. `fits` includes fixed and
fitted parameter values, while `fixed` and `free` preserve their roles.

`fitness` means the selected loss, not necessarily likelihood. `converged`
reports the optimizer's stopping criterion. It does not establish parameter
recovery, a global optimum or an adequate account of the behavior.

In [ ]:
#| code-summary: "0 — Locate the project inputs"

from pathlib import Path

from jaxcmr.helpers import find_project_root, resolve_project_path

try:
    root_path = Path(find_project_root())
except FileNotFoundError:
    root_path = Path.cwd()


In [1]:
#| code-summary: "Inspect the parameter roles and row alignment in a saved fit"

from pathlib import Path
import json

record = json.loads(Path(resolve_project_path("data/fits/Dupertuys2026_CMR_demo.json", root_path)).read_text())
print("Free parameter bounds:", record["free"])
print("Fit-row identifiers:", record["fits"]["subject"])
print("Objectives:", record["fitness"])
print("Optimizer criterion met:", record["converged"])

Free parameter bounds: {'encoding_drift_rate': [2.220446049250313e-16, 0.9999999999999998], 'start_drift_rate': [2.220446049250313e-16, 0.9999999999999998], 'recall_drift_rate': [2.220446049250313e-16, 0.9999999999999998], 'learning_rate': [2.220446049250313e-16, 0.9999999999999998], 'shared_support': [2.220446049250313e-16, 100.0], 'item_support': [2.220446049250313e-16, 100.0], 'primacy_scale': [2.220446049250313e-16, 100.0], 'primacy_decay': [2.220446049250313e-16, 100.0], 'choice_sensitivity': [2.220446049250313e-16, 100.0]}
Fit-row identifiers: [-1]
Objectives: [7202.029122199305]
Optimizer criterion met: [True]


`jaxcmr.summarize.summarize_parameters()` accepts named fit records and
produces a Markdown summary. Specify `query_parameters` to select and order
the displayed parameters. With one pooled fit, its mean, minimum and maximum
all describe the same value; these are not estimates of participant variability.

In [2]:
#| code-summary: "Render the existing parameter-summary helper"

from IPython.display import Markdown, display
from jaxcmr.summarize import summarize_parameters

display(Markdown(summarize_parameters(
    [{**record, "name": "Pooled CMR"}],
    query_parameters=list(record["free"]),
)))

| Parameter | Statistic | Pooled CMR |
|---|---|---|
| fitness | mean | 7202.03 |
|  | min | 7202.03 |
|  | max | 7202.03 |
| encoding drift rate | mean | 0.68 |
|  | min | 0.68 |
|  | max | 0.68 |
| start drift rate | mean | 0.86 |
|  | min | 0.86 |
|  | max | 0.86 |
| recall drift rate | mean | 0.72 |
|  | min | 0.72 |
|  | max | 0.72 |
| learning rate | mean | 0.36 |
|  | min | 0.36 |
|  | max | 0.36 |
| shared support | mean | 22.65 |
|  | min | 22.65 |
|  | max | 22.65 |
| item support | mean | 22.96 |
|  | min | 22.96 |
|  | max | 22.96 |
| primacy scale | mean | 0.08 |
|  | min | 0.08 |
|  | max | 0.08 |
| primacy decay | mean | 15.00 |
|  | min | 15.00 |
|  | max | 15.00 |
| choice sensitivity | mean | 92.97 |
|  | min | 92.97 |
|  | max | 92.97 |


## What the template adds

A `FitResult` is not a complete experiment specification. The fitting
template saves additional fields for the observation source and selection,
model factory, component creators, loss, package version, numerical precision,
source fingerprint and run settings. It also saves the objective recalculated
from the returned parameters.
These are project conventions added around the fitter's result.

[Data Simulation](simulation.ipynb) consumes the returned parameter values
with an explicit assignment to participant IDs. Applying pooled values to
participants requires that assignment; the `-1` label is not a participant
in the observation dataset. [Model Evaluation](evaluation.ipynb) uses the
objective and provenance to establish which comparisons are appropriate.

The [fitting template](../templates/fitting.ipynb) supplies the complete
configuration, optimization, saving, objective recalculation and subsequent
simulation. This guide explains the interfaces that procedure connects.

The supplied CMR fit estimates nine continuous parameters with learning before
context updating and repeated recalls disabled. Preparation compared seeds 0
and 1, each with three restarts; `search_comparison` retains both results and
the selected seed. These are preparation metadata, not required `FitResult`
fields. The source fingerprint identifies the actual code even when a version
label is unchanged. Historical parameters require their original model semantics;
loading them into a changed implementation does not reproduce the original fit.


## Selecting an optimizer in the template

Set both `fit_alg_path` and the complete `fit_hyperparams` value. These short runs
exercise the interface; they are not convergence settings. The notebook adds the
model's free-parameter bounds and seed from `model_configs`.

| `fit_alg_path` | Small `fit_hyperparams` example |
|---|---|
| `jaxcmr.fitting.ScipyDE` | `{"num_steps": 2, "pop_size": 4, "best_of": 1, "polish": False, "init": "latinhypercube", "progress_bar": False}` |
| `jaxcmr.fitting.EvosaxDE` | `{"num_steps": 2, "pop_size": 4, "best_of": 1, "polish": False, "init": "latinhypercube", "progress_bar": False}` |
| `jaxcmr.fitting.ScanEvosaxDE` | `{"num_steps": 2, "pop_size": 4, "best_of": 1, "polish": False, "init": "latinhypercube", "progress_bar": False}` |
| `jaxcmr.fitting.MapEvosaxDE` | `{"num_steps": 2, "pop_size": 4, "best_of": 1, "polish": False, "init": "latinhypercube", "map_batch_size": 1, "progress_bar": False}` |
| `jaxcmr.fitting.ScipyLBFGSB` | `{"num_steps": 2, "best_of": 1, "screening_size": 4, "screening_batch_size": 4, "progress_bar": False}` |
| `jaxcmr.fitting.ScipyTrustConstr` | `{"num_steps": 2, "best_of": 1, "screening_size": 4, "screening_batch_size": 4, "progress_bar": False}` |

DE's `pop_size` multiplies the number of free parameters. `MapEvosaxDE`'s
`map_batch_size` controls participant batching when `pooled=False`; `subject_indices`
still selects the work unit. Gradient fitters require a differentiable objective;
the template's likelihood is suitable. Their screening and restart controls replace
DE's population controls. To exercise only fitting, set `redo_sims=False` and
`redo_figures=False`. One entry in `model_configs` makes a single-model job.
